# Credit Default Prediction and Customer Risk Profiling

## Project Overview

This project uses historical credit card customer data to investigate a practical financial-risk question:

> **Which customer characteristics and repayment behaviors are associated with credit card default, and can these patterns be translated into an interpretable customer risk profile?**

The project emphasizes a complete and understandable data science workflow rather than model complexity:

**Data Understanding → Exploratory Data Analysis → Feature Engineering → Predictive Modeling → Model Evaluation → Model Interpretation → Customer Risk Profiling → Business Conclusion**

### Project Objectives

1. Explore customer credit, repayment, billing, and payment behavior using charts and descriptive statistics.
2. Identify variables that show meaningful statistical associations with default.
3. Build baseline Logistic Regression and Random Forest models.
4. Compare the incremental contribution of demographic, repayment, billing, and payment feature groups.
5. Construct an interpretable, rule-based customer risk profile to examine which observed customer patterns are associated with higher default rates.

### Data and Methodology Note

The dataset is a local copy of the UCI Default of Credit Card Clients dataset. The target variable is `default.payment.next.month`.

The **High Risk** profile used later in the project is a descriptive, rule-based profile. It is not a model prediction label and should not be interpreted as a causal classification. All conclusions are limited to the current dataset and the feature definitions used in this project.


## 1. Data Understanding

**What are we doing here?**

First, we confirm the data source, number of observations and variables, data types, and target variable. This section focuses on understanding the data before moving to modeling.

The dataset is a local copy of the UCI Default of Credit Card Clients dataset. The target variable `default.payment.next.month` is coded as 0 for no default and 1 for default.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)

pd.set_option("display.float_format", "{:.4f}".format)
FIGURE_DIR = Path("../figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Update the local path below if the dataset is stored elsewhere.
DATA_PATHS = [
    Path("../data/default_of_credit_card_clients_dataset.csv"),
    Path(r"D:\Software\MYSQL\dataset\r_dataset\default_of_credit_card_clients_dataset.csv"),
    Path(r"D:\Software\MYSQL\dataset\primary_dataset\default_of_credit_card_clients_dataset.csv")
]

DATA_PATH = next((path for path in DATA_PATHS if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Download the UCI dataset and update DATA_PATHS with the local CSV path."
    )

df = pd.read_csv(DATA_PATH)

print("Data shape:", df.shape)
df.info()
df.head()


In [ ]:
df.describe(include="all")

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_rate": df.isna().mean()
}).sort_values("missing_count", ascending=False)

missing_summary.head(10)

### Data Quality Check

If `missing_count` is zero for all variables, the raw dataset contains no missing values. Median imputation is still kept in the modeling pipeline so that the preprocessing workflow is robust to missing values in future data rather than because the current dataset contains substantial missingness.


## 2. Exploratory Data Analysis (EDA)

**What are we doing here?**

We examine demographic characteristics, credit limits, repayment history, billing behavior, and payment behavior in relation to the default outcome.

The analysis focuses on **descriptive associations**. An observed relationship does not imply a causal relationship.


In [ ]:
target_col = "default.payment.next.month"

target_summary = pd.DataFrame({
    "count": df[target_col].value_counts().sort_index(),
    "proportion": df[target_col].value_counts(normalize=True).sort_index()
})
target_summary

### 2.1 Target Distribution

First, we check whether default and non-default customers are balanced. Because default customers are a minority class, later model evaluation should not rely on Accuracy alone.


In [ ]:
target_plot = target_summary.copy()
target_plot.index = ["Non-default", "Default"]
target_plot["proportion_pct"] = target_plot["proportion"] * 100

target_plot[["count", "proportion_pct"]]

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(target_plot.index, target_plot["proportion_pct"])
plt.ylabel("Share of Customers (%)")
plt.title("Default vs Non-default Customers")
for i, value in enumerate(target_plot["proportion_pct"]):
    plt.text(i, value + 1, f"{value:.1f}%", ha="center")
plt.ylim(0, max(target_plot["proportion_pct"]) + 10)
plt.tight_layout()
plt.show()

**Observation:** Default customers account for about 22% of the sample, indicating a meaningful class imbalance. The modeling stage therefore reports Precision, Recall, F1-score, ROC-AUC, and PR-AUC in addition to Accuracy.


### 2.2 Demographic Characteristics

We compare observed default rates across sex, education, marriage status, and age groups. Small subgroups are interpreted cautiously and are not used to make strong conclusions.


In [ ]:
for col in ["SEX", "EDUCATION", "MARRIAGE"]:
    print(f"\n{col}")
    display(
        df.groupby(col)[target_col]
          .agg(count="count", default_rate="mean")
    )

# Age groups
age_bins = [20, 29, 39, 49, 59, 100]
age_labels = ["21-29", "30-39", "40-49", "50-59", "60+"]
df["AGE-binned"] = pd.cut(
    df["AGE"], bins=age_bins, labels=age_labels, include_lowest=True
)

age_default = df.groupby("AGE-binned", observed=True)[target_col].agg(
    count="count", default_rate="mean"
)
age_default

**Interpretation:** Observed default rates differ across some demographic groups. However, several education and marriage categories contain relatively few observations, so these variables are treated mainly as control features rather than the core business conclusion.


### 2.3 Credit Limit

We first examine the distribution of `LIMIT_BAL`, then divide customers into five quantile groups and compare their observed default rates.


In [ ]:
print(df["LIMIT_BAL"].describe())

plt.figure(figsize=(8, 5))
plt.hist(df["LIMIT_BAL"], bins=30)
plt.xlabel("Credit Limit")
plt.ylabel("Number of Customers")
plt.title("Distribution of Credit Limits")
plt.tight_layout()
plt.show()

df["LIMIT_BAL_group"] = pd.qcut(
    df["LIMIT_BAL"], q=5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
)
credit_default = df.groupby("LIMIT_BAL_group", observed=True)[target_col].agg(
    count="count", default_rate="mean"
)
credit_default

In [ ]:
credit_plot = credit_default.copy()
credit_plot["default_rate_pct"] = credit_plot["default_rate"] * 100

plt.figure(figsize=(8, 5))
plt.bar(credit_plot.index.astype(str), credit_plot["default_rate_pct"])
plt.xlabel("Credit Limit Quintile")
plt.ylabel("Observed Default Rate (%)")
plt.title("Observed Default Rate by Credit Limit Group")
plt.tight_layout()
plt.show()

**Interpretation:** In this dataset, customers in the lower credit-limit groups show higher observed default rates. This should be interpreted as a statistical association rather than evidence that a lower credit limit causes default.


**Observation:** The lowest credit-limit group has the highest observed default rate in the current sample, showing an overall negative association between credit limit and default rate. The analysis does not interpret credit limit as a causal driver of default.


### 2.4 Repayment Behavior

Historical repayment status is one of the most important feature groups in this project. The original analysis printed `PAY_0`–`PAY_6` repeatedly; here they are summarized in one loop, and `delay_months` is constructed as a behavioral feature for later analysis.


In [ ]:
pay_status_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

for col in pay_status_cols:
    print(f"\n{col}")
    display(
        df.groupby(col)[target_col]
          .agg(count="count", default_rate="mean")
    )

df["delay_months"] = (df[pay_status_cols] > 0).sum(axis=1)
delay_default = df.groupby("delay_months")[target_col].agg(
    count="count", default_rate="mean"
)
delay_default

In [ ]:
delay_rate = delay_default["default_rate"]

plt.figure(figsize=(8, 5))
plt.plot(delay_rate.index, delay_rate.values, marker="o")
plt.xlabel("Number of Months with Payment Delay")
plt.ylabel("Default Rate")
plt.title("Default Rate by Number of Months with Payment Delay")
plt.tight_layout()
plt.show()

**Interpretation:** `delay_months` shows a clear positive association with the observed default rate. It is therefore an important EDA variable and a core repayment feature in the predictive models.


### 2.5 Billing Behavior

The original notebook contained repeated distribution checks for `BILL_AMT1`. We retain one representative distribution and use the six monthly bill variables to construct `avg_bill_amt` and `bill_amt_std`.


In [ ]:
print("Negative BILL_AMT1 observations:", (df["BILL_AMT1"] < 0).sum())

plt.figure(figsize=(8, 5))
plt.hist(df["BILL_AMT1"], bins=30)
plt.xlabel("Bill Amount")
plt.ylabel("Frequency")
plt.title("Distribution of BILL_AMT1")
plt.tight_layout()
plt.show()

bill_cols = [
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6"
]

df["avg_bill_amt"] = df[bill_cols].mean(axis=1)
df["bill_amt_std"] = df[bill_cols].std(axis=1)

bill_features_summary = df[["avg_bill_amt", "bill_amt_std"]].describe()
bill_features_summary

In [ ]:
for col in ["avg_bill_amt", "bill_amt_std"]:
    group_col = f"{col}_group"
    df[group_col] = pd.qcut(df[col], q=5, duplicates="drop")
    print(f"\n{col}")
    display(
        df.groupby(group_col, observed=True)[target_col]
          .agg(count="count", default_rate="mean")
    )

**Observation:** Bill amounts are strongly right-skewed and include some negative values. Negative values are not automatically removed because they may represent adjustments or refunds. They are retained in the analysis.


### 2.6 Payment Behavior

The original notebook contained two highly similar histograms for `PAY_AMT1`. We retain one representative plot, truncated at the 99th percentile for visualization, and use the six monthly payment variables to construct average payment and payment volatility.


In [ ]:
payment_cols = [
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
]

p99 = df["PAY_AMT1"].quantile(0.99)
plt.figure(figsize=(8, 5))
plt.hist(df["PAY_AMT1"], bins=50)
plt.xlim(0, p99)
plt.xlabel("PAY_AMT1")
plt.ylabel("Frequency")
plt.title("PAY_AMT1 Distribution (up to 99th percentile)")
plt.tight_layout()
plt.show()

df["avg_payment"] = df[payment_cols].mean(axis=1)
df["payment_std"] = df[payment_cols].std(axis=1)

df[["avg_payment", "payment_std"]].describe()

In [ ]:
for col in ["avg_payment", "payment_std"]:
    group_col = f"{col}_group"
    df[group_col] = pd.qcut(df[col], q=5, duplicates="drop")
    print(f"\n{col}")
    display(
        df.groupby(group_col, observed=True)[target_col]
          .agg(count="count", default_rate="mean")
    )

### 2.7 Payment-to-Bill Ratio

This is a key feature-engineering step. The ratio is not a strict current-period repayment rate. Instead, it is a behavioral proxy based on cumulative six-month payments and bill amounts.

When `total_bill <= 0`, the ratio cannot be meaningfully calculated. Those observations are kept as missing, and a `ratio_missing` indicator is added during modeling.


In [ ]:
df["total_payment"] = df[payment_cols].sum(axis=1)
df["total_bill"] = df[bill_cols].sum(axis=1)

print("Customers with total_bill <= 0:", (df["total_bill"] <= 0).sum())

mask = df["total_bill"] > 0
df["payment_to_bill_ratio"] = np.nan
df.loc[mask, "payment_to_bill_ratio"] = (
    df.loc[mask, "total_payment"] / df.loc[mask, "total_bill"]
)

print(df["payment_to_bill_ratio"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
))
print("Ratio > 5:", (df["payment_to_bill_ratio"] > 5).sum())

df["payment_to_bill_ratio_capped"] = df["payment_to_bill_ratio"].clip(upper=5)

ratio_group = pd.qcut(
    df["payment_to_bill_ratio_capped"], q=5, duplicates="drop"
)
ratio_default = df.groupby(ratio_group, observed=True)[target_col].agg(
    count="count", default_rate="mean"
)
ratio_default

In [ ]:
ratio_plot = ratio_default.copy()
ratio_plot["default_rate_pct"] = ratio_plot["default_rate"] * 100

plt.figure(figsize=(8, 5))
plt.bar(ratio_plot.index.astype(str), ratio_plot["default_rate_pct"])
plt.xlabel("Payment-to-Bill Ratio Group")
plt.ylabel("Observed Default Rate (%)")
plt.title("Observed Default Rate by Payment-to-Bill Ratio")
plt.tight_layout()
plt.show()

**Observation:** A lower payment-to-bill ratio is associated with a higher observed default rate. The feature is therefore retained for both predictive modeling and customer risk profiling.


## 3. Feature Engineering and Modeling Dataset

This section converts the variables already explored in EDA into the formal modeling dataset. The final models use the selected numerical and categorical features rather than temporary EDA grouping variables.

A key data-processing principle is to split the data into training and test sets first. Imputation and scaling are then learned inside the preprocessing Pipeline using training data only. This helps prevent data leakage.


In [ ]:
feature_cols = [
    "LIMIT_BAL",
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    "AGE",
    "delay_months",
    "avg_bill_amt",
    "bill_amt_std",
    "avg_payment",
    "payment_std",
    "payment_to_bill_ratio_capped"
]

df_model = df[feature_cols + [target_col]].copy()
df_model["ratio_missing"] = df_model["payment_to_bill_ratio_capped"].isna().astype(int)

X = df_model.drop(columns=[target_col])
y = df_model[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

In [ ]:
categorical_features = ["SEX", "EDUCATION", "MARRIAGE"]
numeric_features = [
    "LIMIT_BAL", "AGE", "delay_months",
    "avg_bill_amt", "bill_amt_std",
    "avg_payment", "payment_std",
    "payment_to_bill_ratio_capped", "ratio_missing"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

## 4. Predictive Modeling

Two models are used:

- **Logistic Regression:** an interpretable baseline model.
- **Random Forest:** a nonlinear model used for comparison.

Because default customers represent about 22% of the sample, evaluation does not rely on Accuracy alone. Precision, Recall, F1-score, ROC-AUC, and PR-AUC are also reported.


In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ))
])

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

logistic_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

y_prob_lr = logistic_model.predict_proba(X_test)[:, 1]
y_pred_lr = (y_prob_lr >= 0.5).astype(int)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]
y_pred_rf = (y_prob_rf >= 0.5).astype(int)

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob)
    }

model_comparison = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, y_pred_lr, y_prob_lr),
    evaluate_model("Random Forest", y_test, y_pred_rf, y_prob_rf)
])

model_comparison.round(4)

In [ ]:
model_plot = model_comparison.set_index("Model")[
    ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]
]

model_plot.plot(kind="bar", figsize=(10, 5))
plt.ylabel("Score")
plt.title("Model Performance Comparison")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "01_model_performance_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


**How to read the comparison:** Logistic Regression provides higher Recall and F1-score at the default threshold, while Random Forest provides higher Precision and Accuracy. Their ROC-AUC and PR-AUC values are very close, so the analysis does not claim that one model is universally better. The appropriate choice depends on the relative business costs of missed defaults and false alerts.


In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_rf = confusion_matrix(y_test, y_pred_rf)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, cm, title in zip(
    axes,
    [cm_lr, cm_rf],
    ["Logistic Regression", "Random Forest"]
):
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Predicted No Default", "Predicted Default"],
        yticklabels=["Actual No Default", "Actual Default"],
        ax=ax
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted Class")
    ax.set_ylabel("Actual Class")

fig.suptitle("Confusion Matrices at the 0.5 Threshold", y=1.02)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "02_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()


### 4.1 ROC Curve

The two models have very similar ROC-AUC values, suggesting comparable overall ranking ability under the current feature set and train-test split.


In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC={roc_auc_score(y_test, y_prob_lr):.4f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={roc_auc_score(y_test, y_prob_rf):.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "03_roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()


### 4.2 Precision-Recall Curve

Because the positive class represents about 22% of observations, the Precision-Recall curve provides additional information about minority-class performance beyond ROC-AUC.


In [ ]:
precision_lr, recall_lr, _ = precision_recall_curve(y_test, y_prob_lr)
precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_prob_rf)
baseline = y_test.mean()

plt.figure(figsize=(8, 6))
plt.plot(
    recall_lr, precision_lr,
    label=f"Logistic Regression (PR-AUC={average_precision_score(y_test, y_prob_lr):.4f})"
)
plt.plot(
    recall_rf, precision_rf,
    label=f"Random Forest (PR-AUC={average_precision_score(y_test, y_prob_rf):.4f})"
)
plt.axhline(baseline, linestyle="--", label=f"Positive-class prevalence ({baseline:.2%})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "04_precision_recall_curve.png", dpi=300, bbox_inches="tight")
plt.show()


### 4.3 Threshold Analysis

A probability threshold of 0.5 is not the only possible choice. Lowering the threshold generally increases Recall while reducing Precision. In a credit-risk setting, the appropriate threshold should reflect the business cost of false negatives and false positives.


In [ ]:
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
threshold_results = []

for threshold in thresholds:
    pred = (y_prob_lr >= threshold).astype(int)
    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred)
    })

threshold_results = pd.DataFrame(threshold_results)
threshold_results.round(4)

## 5. Model Interpretation

Rather than adding complex explainable-AI methods, this project uses Random Forest feature importance and Logistic Regression coefficients to examine which variables provide substantial information to the models.


In [ ]:
# Random Forest feature importance
rf_preprocessor = rf_model.named_steps["preprocessor"]
rf_classifier = rf_model.named_steps["classifier"]

feature_names = rf_preprocessor.get_feature_names_out()
feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_classifier.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

feature_name_mapping = {
    "num__delay_months": "Number of Delayed Months",
    "num__bill_amt_std": "Bill Amount Volatility",
    "num__payment_to_bill_ratio_capped": "Payment-to-Bill Ratio",
    "num__avg_payment": "Average Payment",
    "num__avg_bill_amt": "Average Bill",
    "num__payment_std": "Payment Volatility",
    "num__LIMIT_BAL": "Credit Limit",
    "num__AGE": "Age",
    "num__ratio_missing": "Missing Ratio Indicator"
}

feature_importance_plot = feature_importance.copy()
feature_importance_plot["feature"] = (
    feature_importance_plot["feature"]
    .map(feature_name_mapping)
    .fillna(feature_importance_plot["feature"])
)

feature_importance_plot.head(10)

In [ ]:
top10 = feature_importance_plot.head(10).sort_values("importance")

plt.figure(figsize=(9, 6))
plt.barh(top10["feature"], top10["importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "05_random_forest_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()


### Logistic Regression Coefficients

Continuous variables have been standardized. Therefore, their coefficients and odds ratios can be interpreted in terms of an approximately one-standard-deviation increase, holding the other model features constant. Categorical variables must be interpreted relative to their reference categories.


In [ ]:
lr_preprocessor = logistic_model.named_steps["preprocessor"]
lr_classifier = logistic_model.named_steps["classifier"]

lr_feature_names = lr_preprocessor.get_feature_names_out()
coef_df = pd.DataFrame({
    "feature": lr_feature_names,
    "coefficient": lr_classifier.coef_[0]
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df["odds_ratio"] = np.exp(coef_df["coefficient"])

coef_df.sort_values("abs_coefficient", ascending=False).head(15)

## 6. Feature Group Contribution

This section addresses a practical question: **How much additional information is gained when repayment, billing, and payment variables are added to demographic and credit variables?**

The original notebook contained repeated `X_train_v2 / v3 / v4`, preprocessing, and model code. The revised version uses a small reusable function while preserving the original four feature-group designs.


In [ ]:
baseline_features = ["AGE", "SEX", "EDUCATION", "MARRIAGE", "LIMIT_BAL"]
repayment_features = ["delay_months", "payment_to_bill_ratio_capped", "ratio_missing"]
billing_features = ["avg_bill_amt", "bill_amt_std"]
payment_features = ["avg_payment", "payment_std"]

feature_sets = {
    "Demographic + Credit": baseline_features,
    "+ Repayment Behavior": baseline_features + repayment_features,
    "+ Billing Behavior": baseline_features + repayment_features + billing_features,
    "+ Payment Behavior": baseline_features + repayment_features + billing_features + payment_features
}

def build_logistic_pipeline(features):
    categorical = [c for c in ["SEX", "EDUCATION", "MARRIAGE"] if c in features]
    numeric = [c for c in features if c not in categorical]
    prep = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical)
    ])
    return Pipeline([
        ("preprocessor", prep),
        ("classifier", LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=42
        ))
    ])

feature_group_rows = []
for name, features in feature_sets.items():
    model = build_logistic_pipeline(features)
    model.fit(X_train[features], y_train)
    prob = model.predict_proba(X_test[features])[:, 1]
    pred = (prob >= 0.5).astype(int)
    feature_group_rows.append({
        "Feature Set": name,
        "ROC-AUC": roc_auc_score(y_test, prob),
        "F1": f1_score(y_test, pred)
    })

feature_group_results = pd.DataFrame(feature_group_rows)
feature_group_results["Delta ROC-AUC"] = feature_group_results["ROC-AUC"].diff()
feature_group_results.round(4)

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    feature_group_results["Feature Set"],
    feature_group_results["ROC-AUC"],
    marker="o", linewidth=2
)
plt.ylabel("ROC-AUC")
plt.xlabel("Feature Set")
plt.title("Predictive Performance with Incremental Feature Groups")
plt.ylim(0.55, 0.80)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "06_feature_group_contribution.png", dpi=300, bbox_inches="tight")
plt.show()


**Key interpretation:** Based on the project results, adding repayment behavior increases ROC-AUC from approximately 0.603 to approximately 0.741, representing the largest incremental improvement. Billing behavior adds very little additional information after repayment variables are included, while payment behavior provides a further but more modest improvement.

This provides a more complete conclusion than simply stating that `delay_months` is the most important individual feature because it evaluates the contribution of whole feature groups.


## 7. Customer Risk Profiling

The model results are translated into an easier-to-understand customer segmentation using `delay_months` and `payment_to_bill_ratio`. Customers are grouped into Low, Medium, and High Risk profiles, with an additional Unknown category when the ratio is unavailable.

This is **rule-based descriptive profiling**, not a model prediction label. It is used to summarize repayment patterns and their observed default rates.


In [ ]:
df["delay_group"] = pd.cut(
    df["delay_months"],
    bins=[-1, 0, 2, 6],
    labels=["Low", "Medium", "High"]
)

df["payment_ratio_group"] = pd.qcut(
    df["payment_to_bill_ratio_capped"],
    q=3,
    labels=["Low", "Medium", "High"]
)

risk_pattern = (
    df.groupby(["delay_group", "payment_ratio_group"], observed=True)[target_col]
      .agg(customer_count="count", default_rate="mean")
      .reset_index()
)
risk_pattern["default_rate"] *= 100
risk_pattern.round(2)

In [ ]:
risk_heatmap = risk_pattern.pivot(
    index="delay_group",
    columns="payment_ratio_group",
    values="default_rate"
)

plt.figure(figsize=(7, 4.5))
sns.heatmap(
    risk_heatmap,
    annot=True,
    fmt=".1f",
    cmap="RdYlBu_r",
    cbar_kws={"label": "Observed Default Rate (%)"}
)
plt.xlabel("Payment-to-Bill Ratio Group")
plt.ylabel("Payment Delay Group")
plt.title("Observed Default Rate by Repayment Behavior")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "07_repayment_behavior_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


**Key observation:** Payment delay and payment-to-bill behavior provide complementary information. Among customers with more repayment problems, differences in payment-to-bill behavior are still associated with different observed default rates. This supports using both variables in the customer profile rather than relying on delayed months alone.


In [ ]:
df["repayment_risk_profile"] = "Medium Risk"

ratio_missing = df["payment_ratio_group"].isna()

df.loc[
    (~ratio_missing) &
    (df["delay_group"] == "Low") &
    (df["payment_ratio_group"].isin(["Medium", "High"])),
    "repayment_risk_profile"
] = "Low Risk"

df.loc[
    (~ratio_missing) &
    (df["delay_group"] == "High") &
    (df["payment_ratio_group"].isin(["Low", "Medium"])),
    "repayment_risk_profile"
] = "High Risk"

df.loc[ratio_missing, "repayment_risk_profile"] = "Unknown"

profile_summary = (
    df.groupby("repayment_risk_profile", observed=False)
      .agg(
          customer_count=(target_col, "size"),
          default_rate=(target_col, "mean"),
          avg_age=("AGE", "mean"),
          avg_credit_limit=("LIMIT_BAL", "mean"),
          avg_bill=("avg_bill_amt", "mean"),
          avg_payment=("avg_payment", "mean"),
          avg_payment_ratio=("payment_to_bill_ratio_capped", "mean"),
          avg_delay_months=("delay_months", "mean")
      )
      .reset_index()
)
profile_summary["default_rate"] *= 100
profile_summary.round(2)

**Why is there an Unknown group?**

There are 1,071 customers for whom the cumulative bill amount does not satisfy the condition required to calculate a meaningful payment-to-bill ratio. These customers should not be forced into the Medium Risk group. Keeping them as Unknown avoids an artificial risk classification.


### 7.1 Financial Scale Proxy

To examine whether repayment profiles show similar patterns across different levels of financial scale, an exploratory **Financial Scale Proxy** is constructed from `LIMIT_BAL` and `avg_bill_amt`.

This is not a standard financial metric. It is used only for internal comparison within this project.


In [ ]:
scale_cols = ["LIMIT_BAL", "avg_bill_amt"]
scale_values = StandardScaler().fit_transform(df[scale_cols])
df["financial_scale_score"] = scale_values.mean(axis=1)
df["financial_scale"] = pd.qcut(
    df["financial_scale_score"], q=3, labels=["Low", "Medium", "High"]
)

customer_profile = (
    df.groupby(["financial_scale", "repayment_risk_profile"], observed=True)[target_col]
      .agg(customer_count="count", default_rate="mean")
      .reset_index()
)
customer_profile["default_rate"] *= 100
customer_profile.round(2)

### 7.2 Risk Profile Heatmap

To compare variables with different units on one chart, the selected metrics for the three Risk Profiles are standardized using z-scores. **The colors represent relative levels across the three profiles, not raw monetary amounts.**


In [ ]:
risk_order = ["Low Risk", "Medium Risk", "High Risk"]

risk_profile_heatmap = (
    profile_summary[profile_summary["repayment_risk_profile"].isin(risk_order)]
    .set_index("repayment_risk_profile")
    .loc[risk_order]
)

heatmap_data = risk_profile_heatmap[[
    "default_rate", "avg_credit_limit", "avg_bill",
    "avg_payment", "avg_payment_ratio", "avg_delay_months"
]]

heatmap_scaled = pd.DataFrame(
    StandardScaler().fit_transform(heatmap_data),
    index=heatmap_data.index,
    columns=heatmap_data.columns
)

feature_labels = {
    "default_rate": "Default Rate",
    "avg_credit_limit": "Credit Limit",
    "avg_bill": "Average Bill",
    "avg_payment": "Average Payment",
    "avg_payment_ratio": "Payment-to-Bill Ratio",
    "avg_delay_months": "Average Delay Months"
}

heatmap_plot = heatmap_scaled.rename(columns=feature_labels)

plt.figure(figsize=(10, 5.5))
sns.heatmap(
    heatmap_plot,
    annot=heatmap_plot.round(2),
    fmt=".2f",
    cmap="RdYlBu_r",
    center=0,
    linewidths=0.5,
    cbar_kws={"label": "Standardized value (z-score)"}
)
plt.title("Customer Risk Profiles: Financial and Repayment Characteristics")
plt.xlabel("Customer Characteristics")
plt.ylabel("Repayment Risk Profile")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "08_customer_risk_profile_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
profile_plot = profile_summary[
    profile_summary["repayment_risk_profile"].isin(["Low Risk", "Medium Risk", "High Risk"])
].copy()

profile_plot = profile_plot.set_index("repayment_risk_profile").loc[
    ["Low Risk", "Medium Risk", "High Risk"]
]

plt.figure(figsize=(8, 5))
plt.bar(profile_plot.index, profile_plot["default_rate"])
plt.ylabel("Observed Default Rate (%)")
plt.xlabel("Repayment Risk Profile")
plt.title("Observed Default Rate by Customer Risk Profile")
for i, value in enumerate(profile_plot["default_rate"]):
    plt.text(i, value + 2, f"{value:.1f}%", ha="center")
plt.ylim(0, max(profile_plot["default_rate"]) + 12)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "09_customer_risk_profile_default_rate.png", dpi=300, bbox_inches="tight")
plt.show()


## 7.3 What type of customer is more likely to default?

Based on the descriptive risk profile, customers with the following **observed characteristics** are more likely to belong to the higher-default group:

- multiple months with payment delays;
- a relatively low payment-to-bill ratio;
- lower average payment amounts;
- relatively lower credit limits within this dataset.

In the current sample, the rule-based **High Risk** profile has an observed default rate of about **60.9%**, compared with about **11.2%** for the Low Risk profile. The High Risk group also has substantially more delayed months and a much lower payment-to-bill ratio.

This should be interpreted as a **customer pattern**, not a causal statement. The risk profile is constructed from historical repayment behavior and is intended to summarize patterns in the dataset rather than make a real-world credit decision.


## 7.4 Customer Profile Summary

| Profile | Typical observed characteristics | Observed default rate |
|---|---|---:|
| **Low Risk** | No positive payment-delay months; higher payment-to-bill ratio; higher average payment | ~11.2% |
| **Medium Risk** | Some payment delays and intermediate payment-to-bill ratio | ~21.9% |
| **High Risk** | More payment delays; low payment-to-bill ratio; lower average payment | ~60.9% |
| **Unknown** | Payment-to-bill ratio unavailable | ~34.4% |

The **Unknown** group is kept separate because its payment-to-bill ratio cannot be meaningfully calculated. It should not be treated as a fourth risk tier.


## 8. Key Findings

### 1. Repayment behavior is the strongest incremental information source

Feature-group analysis shows that adding repayment variables produces the largest increase in ROC-AUC. This indicates that historical repayment behavior contains substantial information for distinguishing default outcomes in this dataset.

### 2. Payment behavior adds useful information beyond delay counts

The payment-to-bill ratio and payment-related variables provide additional differentiation among customers with different repayment patterns. This is why the project does not rely on `delay_months` alone.

### 3. The customer risk profile reveals a clear behavioral pattern

The High Risk profile is characterized by repeated payment delays and a low payment-to-bill ratio, and it has a substantially higher observed default rate than the Low Risk profile.

### 4. Model performance involves a precision-recall trade-off

Logistic Regression and Random Forest have similar ROC-AUC and PR-AUC, but their Precision and Recall differ at the default threshold. This shows why model selection should consider the purpose of the analysis rather than Accuracy alone.

### 5. The project connects prediction with customer-level interpretation

Instead of stopping at model metrics, the analysis translates statistical and machine-learning results into an interpretable customer profile. This creates an end-to-end data science workflow relevant to financial risk analysis.


## 9. Limitations

1. **Single dataset:** Results may not generalize to other institutions or customer populations.
2. **Observational analysis:** Associations should not be interpreted as causal relationships.
3. **Feature construction:** `delay_months` and `payment_to_bill_ratio` are constructed proxies rather than direct measures of financial capacity.
4. **Model scope:** The project intentionally focuses on Logistic Regression and Random Forest rather than adding advanced models only for complexity.
5. **Class imbalance:** Default cases are a minority, so Accuracy alone is insufficient.
6. **Validation scope:** The current analysis uses one stratified train-test split. Cross-validation, calibration, fairness analysis, and external validation would be needed before practical deployment.


## 10. Conclusion

This project examined whether demographic, credit, billing, and payment behavior could be used to understand and predict credit card default.

The analysis found that **repayment behavior provides the strongest incremental predictive information**, while payment behavior provides additional information. The customer profiling analysis further shows a clear observed pattern: customers with repeated payment delays and lower payment-to-bill ratios have substantially higher default rates in this dataset.

The project therefore combines three levels of analysis:

1. **Descriptive analysis:** What does the customer data look like?
2. **Predictive analysis:** Can customer characteristics help distinguish default outcomes?
3. **Customer profiling:** What observable repayment patterns characterize groups with different default rates?

The result is an interpretable financial-risk analysis rather than a claim of causal credit-risk prediction.


## 11. Future Work

The current project is intentionally scoped as a practical and interpretable first data science project. Possible extensions include:

- cross-validation to assess model stability;
- probability calibration for more reliable risk probabilities;
- threshold selection based on explicit business costs;
- comparison with a small number of gradient-boosting models;
- individual-level explainability such as SHAP when computationally appropriate;
- subgroup performance and fairness analysis;
- validation on an external credit-risk dataset.

These are proposed extensions and are **not presented as completed work** in this project.


---

# Portfolio Summary

**Project theme:** Credit Default Prediction and Customer Risk Profiling  
**Domain:** Financial Risk / Credit Analytics  
**Tools:** Python, pandas, NumPy, matplotlib, seaborn, scikit-learn  
**Methods:** EDA, feature engineering, Logistic Regression, Random Forest, ROC-AUC, PR-AUC, threshold analysis, feature importance, customer segmentation

**Main takeaway:** Historical repayment behavior—especially payment delays and payment-to-bill behavior—shows the strongest relationship with default outcomes in this dataset. The project converts these findings into an interpretable customer risk profile while clearly separating descriptive association from causal claims.


## 12. Portfolio Results Snapshot

The repository also includes selected exported figures under `figures/` so that the main results can be reviewed without executing the notebook.

### Model discrimination

| Model | ROC-AUC | PR-AUC | F1 | Precision | Recall |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.7505 | 0.4901 | 0.5032 | 0.4534 | 0.5652 |
| Random Forest | 0.7494 | 0.4914 | 0.3874 | 0.5984 | 0.2864 |

The ROC curves are generated by the modeling section when the dataset is available. The repository README uses a stable ROC-AUC summary image so that the headline result remains viewable without shipping the raw dataset.

### Selected figures

![Model performance comparison](../figures/01_model_performance_comparison.png)

![Confusion matrices](../figures/02_confusion_matrices.png)

![Random Forest feature importance](../figures/05_random_forest_feature_importance.png)

![Feature-group contribution](../figures/06_feature_group_contribution.png)

![Customer risk profile](../figures/09_customer_risk_profile_default_rate.png)
